# **Preamble**

In [1]:
pip install soccerdata pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

# -------------------------------------------------------
# PATH CONFIG
# Notebooks live at Notebooks/<subfolder>/ so project
# root is two levels up from os.getcwd()
# -------------------------------------------------------
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
INPUTS  = os.path.join(PROJECT_ROOT, "Inputs")
OUTPUTS = os.path.join(PROJECT_ROOT, "Outputs")
from pathlib import Path
SOCCERDATA_CACHE = Path(INPUTS) / "soccerdata_cache"


In [3]:
import soccerdata as sd
import pandas as pd
import os

[05/17/26 16:33:37] INFO     No custom team name replacements found. You can configure these in       ]8;id=15574453;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=15574454;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_config.py#92\92]8;;\
                             /Users/patrickknott/soccerdata/config/teamname_replacements.json.                     

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=15574460;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=15574461;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_config.py#190\190]8;;\
                             /Users/patrickknott/soccerdata/config/league_dict.json.                               

In [4]:
import time

# -------------------------------------------------------
# RETRY UTILITY — used by all scraping cells below
# -------------------------------------------------------
MAX_RETRIES = 3
RETRY_DELAY = 15  # seconds; multiplied by attempt number

def with_retry(fn, *args, retries=MAX_RETRIES, backoff=RETRY_DELAY, **kwargs):
    """Call fn(*args, **kwargs), retrying with linear backoff on any exception."""
    for attempt in range(1, retries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            if attempt == retries:
                print(f"  All {retries} attempts failed.")
                raise
            wait = backoff * attempt
            print(f"  Attempt {attempt}/{retries} failed ({type(e).__name__}: {e}). Retrying in {wait}s...")
            time.sleep(wait)


# **Previous Seasons**

In [5]:
# ----------------------------
# CONFIG
# ----------------------------
SEASONS = [
    "2014/2015","2015/2016","2016/2017","2017/2018","2018/2019",
    "2019/2020","2020/2021","2021/2022","2022/2023","2023/2024","2024/2025",
]

# folder_name : (soccerdata league code, file prefix)
LEAGUES = {
    "Prem":  ("ENG-Premier League", "E0"),
    "Liga":  ("ESP-La Liga", "SP1"),
    "Bund":  ("GER-Bundesliga", "D1"),
    "Serie": ("ITA-Serie A", "I1"),
    "Ligue": ("FRA-Ligue 1", "F1"),
}

BASE = INPUTS

# ----------------------------
# RUN
# ----------------------------
for folder_name, (league_code, file_prefix) in LEAGUES.items():
    out_dir  = os.path.join(BASE, folder_name, "Previous Seasons")
    temp_dir = os.path.join(out_dir, "_temp")
    os.makedirs(temp_dir, exist_ok=True)

    out_csv = os.path.join(out_dir, f"{file_prefix} Previous Seasons.csv")

    print(f"\n=== {folder_name} ({league_code}) ===")
    frames = []

    for season in SEASONS:
        season_slug = season.replace("/", "_")
        temp_csv    = os.path.join(temp_dir, f"{season_slug}.csv")

        if os.path.exists(temp_csv):
            print(f"  ✓ Cached: {season}")
            df = pd.read_csv(temp_csv)
        else:
            print(f"  Fetching {season} ...")
            us = with_retry(sd.Understat, leagues=league_code, seasons=season, data_dir=SOCCERDATA_CACHE)
            df = with_retry(us.read_team_match_stats)
            if "season" not in df.columns:
                df["season"] = season
            df.to_csv(temp_csv, index=False)

        frames.append(df)

    merged = pd.concat(frames, ignore_index=True)

    for date_col in ["date", "match_date"]:
        if date_col in merged.columns:
            merged = merged.sort_values(date_col).reset_index(drop=True)
            break

    merged.to_csv(out_csv, index=False)
    print(f"\u2705 Saved -> {out_csv}")
    print("   Shape:", merged.shape)
    print("   (Fetched seasons cached in _temp/ — re-run resumes from last failure)")



=== Prem (ENG-Premier League) ===
  Fetching 2014/2015 ...


                    INFO     Saving cached data to                                                   ]8;id=13003101;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=13003102;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\
                             /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/soccerda               
                             ta_cache                                                                              

[2026-05-17 15:56:31] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: /Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/tls_requests/bin/tls-client-darwin-arm64-1.13.1.dylib


                    INFO     Successfully loaded TLS library:                                      ]8;id=13003109;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/tls_requests/models/libraries.py\libraries.py]8;;\:]8;id=13003110;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/tls_requests/models/libraries.py#397\397]8;;\
                             /Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.1                 
                             2/site-packages/tls_requests/bin/tls-client-darwin-arm64-1.13.1.dylib                 

  Fetching 2015/2016 ...


...

  Fetching 2023/2024 ...


                    INFO     Saving cached data to                                                   ]8;id=13003427;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=13003428;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\
                             /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/soccerda               
                             ta_cache                                                                              

  Fetching 2024/2025 ...


                    INFO     Saving cached data to                                                   ]8;id=13003433;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=13003434;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\
                             /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/soccerda               
                             ta_cache                                                                              

✅ Saved -> /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/Ligue/Previous Seasons/F1 Previous Seasons.csv
   Shape: (3931, 27)
   (Fetched seasons cached in _temp/ — re-run resumes from last failure)


# **Current Season**

In [6]:
# ----------------------------
# CONFIG
# ----------------------------
SEASON = "2025/2026"

# folder_name : (soccerdata league code, file prefix)
LEAGUES = {
    "Prem":  ("ENG-Premier League", "E0"),
    "Liga":  ("ESP-La Liga", "SP1"),
    "Bund":  ("GER-Bundesliga", "D1"),
    "Serie": ("ITA-Serie A", "I1"),
    "Ligue": ("FRA-Ligue 1", "F1"),
}

BASE = INPUTS

# ----------------------------
# RUN
# ----------------------------
for folder_name, (league_code, file_prefix) in LEAGUES.items():
    out_dir = os.path.join(BASE, folder_name, "Current Season")
    os.makedirs(out_dir, exist_ok=True)

    out_csv = os.path.join(out_dir, f"{file_prefix} Current Season.csv")

    print(f"\n=== {folder_name} ({league_code}) {SEASON} ===")

    us = with_retry(sd.Understat, leagues=league_code, seasons=SEASON, data_dir=SOCCERDATA_CACHE)
    df = with_retry(us.read_team_match_stats)

    if "season" not in df.columns:
        df["season"] = SEASON

    for date_col in ["date", "match_date"]:
        if date_col in df.columns:
            df = df.sort_values(date_col).reset_index(drop=True)
            break

    df.to_csv(out_csv, index=False)
    print(f"\u2705 Saved -> {out_csv}")
    print("   Shape:", df.shape)



=== Prem (ENG-Premier League) 2025/2026 ===


[05/17/26 15:57:48] INFO     Saving cached data to                                                   ]8;id=13003439;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=13003440;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\
                             /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/soccerda               
                             ta_cache                                                                              

✅ Saved -> /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/Prem/Current Season/E0 Current Season.csv
   Shape: (362, 27)

=== Liga (ESP-La Liga) 2025/2026 ===


                    INFO     Saving cached data to                                                   ]8;id=13003445;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=13003446;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\
                             /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/soccerda               
                             ta_cache                                                                              

✅ Saved -> /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/Liga/Current Season/SP1 Current Season.csv
   Shape: (360, 27)

=== Bund (GER-Bundesliga) 2025/2026 ===


...

✅ Saved -> /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/Bund/Current Season/D1 Current Season.csv
   Shape: (306, 27)

=== Serie (ITA-Serie A) 2025/2026 ===


[05/17/26 15:57:51] INFO     Saving cached data to                                                   ]8;id=13003457;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=13003458;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\
                             /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/soccerda               
                             ta_cache                                                                              

✅ Saved -> /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/Serie/Current Season/I1 Current Season.csv
   Shape: (365, 27)

=== Ligue (FRA-Ligue 1) 2025/2026 ===


[05/17/26 15:57:52] INFO     Saving cached data to                                                   ]8;id=13003463;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=13003464;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\
                             /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/soccerda               
                             ta_cache                                                                              

✅ Saved -> /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/Ligue/Current Season/F1 Current Season.csv
   Shape: (297, 27)


# **Fixtures**

In [7]:
# =========================================================
# Build ESPN -> Understat team-name mapping
# =========================================================

import soccerdata as sd
import pandas as pd
import os
import re
import unicodedata
from difflib import SequenceMatcher

# ---------------------------------------------------------
# CONFIG — edit here only
# ---------------------------------------------------------
MODE              = "map_all"    # "map_current" or "map_all"
CURRENT_SEASON    = "2025/26"
ALL_SEASONS_START = "2014/15"

OUTPUT_DIR = os.path.join(INPUTS, "Mapping")
os.makedirs(OUTPUT_DIR, exist_ok=True)

LEAGUES = {
    "ENG-Premier League": "eng.1",
    "GER-Bundesliga":     "ger.1",
    "ITA-Serie A":        "ita.1",
    "ESP-La Liga":        "esp.1",
    "FRA-Ligue 1":        "fra.1",
}

# Hardcoded overrides — ESPN name : Understat name
OVERRIDES = {
    "AFC Bournemouth":        "Bournemouth",
    "Brighton & Hove Albion": "Brighton",
}

# Remove these from Understat pool before matching
UNDERSTAT_EXCLUDE = {
    "ITA-Serie A": {"Carpi", "Parma"},
}

# ---------------------------------------------------------
# HELPERS
# ---------------------------------------------------------
def normalise_season(season_str: str) -> str:
    return season_str.strip().replace("/", "-")

def season_range(start: str, end: str) -> list:
    def to_year(s):
        return int(s.strip().replace("/", "-").split("-")[0])
    start_year = to_year(start)
    end_year   = to_year(end)
    seasons = []
    for y in range(start_year, end_year + 1):
        short = str(y + 1)[-2:]
        seasons.append(f"{y}-{short}")
    return seasons

def clean_schedule(df: pd.DataFrame) -> pd.DataFrame:
    df = df.reset_index()
    df["date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")
    df = df.dropna(subset=["date", "home_team", "away_team"]).copy()
    df = df.sort_values(["date"]).reset_index(drop=True)
    return df[["date", "home_team", "away_team"]]

def strip_accents(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in text if not unicodedata.combining(ch))

def normalise_name(name: str) -> str:
    name = strip_accents(name).lower()
    replacements = {"&": " and ", ".": " ", "-": " ", "'": ""}
    for old, new in replacements.items():
        name = name.replace(old, new)
    stopwords = {
        "fc", "cf", "ac", "afc", "as", "ssc", "sc", "sv",
        "vfb", "vfl", "tsg", "fsv", "rc", "stade", "1", "1846",
        "calcio", "2013", "1913",
    }
    tokens = re.findall(r"[a-z0-9]+", name)
    tokens = [t for t in tokens if t not in stopwords]
    token_map = {
        "monchengladbach": "mgladbach",
        "moenchengladbach": "mgladbach",
        "rasenballsport":  "rb",
        "internazionale":  "inter",
        "saint":           "st",
        "hellas":          "",
    }
    tokens = [token_map.get(t, t) for t in tokens]
    tokens = [t for t in tokens if t]
    return " ".join(tokens)

def similarity(a: str, b: str) -> float:
    a_norm = normalise_name(a)
    b_norm = normalise_name(b)
    seq_score   = SequenceMatcher(None, a_norm, b_norm).ratio()
    a_tokens    = set(a_norm.split())
    b_tokens    = set(b_norm.split())
    token_score = len(a_tokens & b_tokens) / len(a_tokens | b_tokens) if (a_tokens or b_tokens) else 0.0
    return 0.65 * seq_score + 0.35 * token_score

def build_team_mapping(espn_teams, understat_teams):
    """
    Understat-led matching — iterate over Understat teams as source of truth.
    ESPN orphans are ignored. Teams with no ESPN match get source='understat_only'.
    """
    unmatched_espn = set(espn_teams)
    mapping_rows   = []

    # Apply overrides first
    for espn_name, us_name in OVERRIDES.items():
        if espn_name in unmatched_espn and us_name in understat_teams:
            mapping_rows.append({
                "espn_team":      espn_name,
                "understat_team": us_name,
                "score":          1.0,
                "source":         "override",
            })
            unmatched_espn.discard(espn_name)

    # Fuzzy match — Understat-led
    unmatched_us_teams = set(understat_teams) - {r["understat_team"] for r in mapping_rows}

    for us_team in sorted(unmatched_us_teams):
        if not unmatched_espn:
            mapping_rows.append({
                "espn_team":      None,
                "understat_team": us_team,
                "score":          None,
                "source":         "understat_only",
            })
            continue

        candidates = [(e, similarity(us_team, e)) for e in unmatched_espn]
        candidates.sort(key=lambda x: x[1], reverse=True)
        best_espn, best_score = candidates[0]

        if best_score < 0.35:
            mapping_rows.append({
                "espn_team":      None,
                "understat_team": us_team,
                "score":          None,
                "source":         "understat_only",
            })
        else:
            mapping_rows.append({
                "espn_team":      best_espn,
                "understat_team": us_team,
                "score":          round(best_score, 4),
                "source":         "fuzzy",
            })
            unmatched_espn.discard(best_espn)

    return pd.DataFrame(mapping_rows).sort_values(
        ["score", "espn_team"], ascending=[False, True],
        na_position="last"
    ).reset_index(drop=True)

def get_teams_for_season(league, season):
    def _espn():
        return clean_schedule(sd.ESPN(leagues=league, seasons=season, data_dir=SOCCERDATA_CACHE).read_schedule())
    def _us():
        return clean_schedule(sd.Understat(leagues=league, seasons=season, data_dir=SOCCERDATA_CACHE).read_schedule())
    try:
        espn_sched = with_retry(_espn)
        us_sched   = with_retry(_us)
        espn_teams = set(espn_sched["home_team"]).union(set(espn_sched["away_team"]))
        us_teams   = set(us_sched["home_team"]).union(set(us_sched["away_team"]))
        print(f"    OK: {league} {season} — ESPN {len(espn_teams)} teams, Understat {len(us_teams)} teams")
        return espn_teams, us_teams
    except Exception as e:
        print(f"    FAILED after retries: {league} {season} — {e}")
        return set(), set()

# ---------------------------------------------------------
# MAIN
# ---------------------------------------------------------
current_season = normalise_season(CURRENT_SEASON)

if MODE == "map_current":
    seasons_to_pull = [current_season]
    out_filename    = f"Club_Mapping_{current_season}.csv"
else:
    seasons_to_pull = season_range(ALL_SEASONS_START, CURRENT_SEASON)
    out_filename    = "Club_Mapping_All.csv"

print(f"Mode:    {MODE}")
print(f"Seasons: {seasons_to_pull}")
print(f"Output:  {out_filename}\n")

all_mapping_dfs = []

for league in LEAGUES:
    print(f"\nProcessing {league}...")

    espn_teams_all = set()
    us_teams_all   = set()

    for season in seasons_to_pull:
        espn_teams, us_teams = get_teams_for_season(league, season)
        espn_teams_all |= espn_teams
        us_teams_all   |= us_teams

    if league in UNDERSTAT_EXCLUDE:
        us_teams_all -= UNDERSTAT_EXCLUDE[league]

    print(f"  Total unique ESPN teams:      {len(espn_teams_all)}")
    print(f"  Total unique Understat teams: {len(us_teams_all)}")

    if not espn_teams_all or not us_teams_all:
        print(f"  Skipping {league} — no data")
        continue

    mapping_df = build_team_mapping(espn_teams_all, us_teams_all)
    mapping_df.insert(0, "league", league)
    all_mapping_dfs.append(mapping_df)

    print(f"\n  Mappings for {league}:")
    print(mapping_df.sort_values("espn_team").to_string(index=False))

# ---------------------------------------------------------
# SAVE
# ---------------------------------------------------------
final_df = pd.concat(all_mapping_dfs, ignore_index=True)
out_path = os.path.join(OUTPUT_DIR, out_filename)
final_df.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")
print(f"Total mappings: {len(final_df)}")

# Understat-only teams (no ESPN match)
understat_only = final_df[final_df["source"] == "understat_only"]
if not understat_only.empty:
    print(f"\nTeams with no ESPN match — will use Understat fixture data ({len(understat_only)}):")
    print(understat_only[["league", "understat_team"]].to_string(index=False))

# Low confidence check
low_conf = final_df[(final_df["score"].notna()) & (final_df["score"] < 0.75)]
if not low_conf.empty:
    print(f"\nLow-confidence mappings to review ({len(low_conf)} rows):")
    print(low_conf.sort_values(["league", "score"]).to_string(index=False))
else:
    print("\nNo low-confidence mappings flagged.")

Mode:    map_all
Seasons: ['2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
Output:  Club_Mapping_All.csv


Processing ENG-Premier League...


[05/17/26 15:57:53] INFO     Saving cached data to                                                   ]8;id=13003469;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=13003470;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\
                             /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/soccerda               
                             ta_cache                                                                              

[05/17/26 15:58:21] INFO     Saving cached data to                                                   ]8;id=13003475;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=13003476;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\
                             /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/soccerda               
                             ta_cache                                                                              

    OK: ENG-Premier League 2014-15 — ESPN 20 teams, Understat 20 teams


                    INFO     Saving cached data to                                                   ]8;id=13003481;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=13003482;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\
                             /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/soccerda               
                             ta_cache                                                                              

...

[05/17/26 16:25:58] INFO     Saving cached data to                                                   ]8;id=13004171;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=13004172;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\
                             /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/soccerda               
                             ta_cache                                                                              

    OK: FRA-Ligue 1 2024-25 — ESPN 18 teams, Understat 18 teams


[05/17/26 16:25:59] INFO     Saving cached data to                                                   ]8;id=13004177;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=13004178;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\
                             /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/soccerda               
                             ta_cache                                                                              

[05/17/26 16:26:22] INFO     Saving cached data to                                                   ]8;id=13004183;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=13004184;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\
                             /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/soccerda               
                             ta_cache                                                                              

    OK: FRA-Ligue 1 2025-26 — ESPN 18 teams, Understat 18 teams
  Total unique ESPN teams:      34
  Total unique Understat teams: 34

  Mappings for FRA-Ligue 1:
     league             espn_team        understat_team  score source
FRA-Ligue 1            AC Ajaccio               Ajaccio 1.0000  fuzzy
FRA-Ligue 1            AJ Auxerre               Auxerre 0.7103  fuzzy
FRA-Ligue 1             AS Monaco                Monaco 1.0000  fuzzy
FRA-Ligue 1     AS Nancy Lorraine                 Nancy 0.5171  fuzzy
FRA-Ligue 1                Angers                Angers 1.0000  fuzzy
FRA-Ligue 1                Bastia             SC Bastia 1.0000  fuzzy
FRA-Ligue 1              Bordeaux              Bordeaux 1.0000  fuzzy
FRA-Ligue 1                 Brest                 Brest 1.0000  fuzzy
FRA-Ligue 1                  Caen                  Caen 1.0000  fuzzy
FRA-Ligue 1         Clermont Foot         Clermont Foot 1.0000  fuzzy
FRA-Ligue 1             Dijon FCO                 Dijon 0.6393  fuz

In [5]:
# =================================================
# Pull fixtures from ESPN for a custom date range,
# map team names from ESPN -> Understat using saved CSV,
# and save one CSV per league
# =================================================

import soccerdata as sd
import pandas as pd
import os

# -----------------------------
# USER CONFIG
# -----------------------------
season = "2025-26"

# Enter dates in dd-mm-yyyy format
start_date_input = "15-05-2026"
end_date_input   = "19-05-2026"

# Mapping CSV created earlier
# Prefer the all-seasons mapping if available, fall back to season-specific
_map_all  = os.path.join(INPUTS, "Mapping", "Club_Mapping_All.csv")
_map_curr = os.path.join(INPUTS, "Mapping", f"Club_Mapping_{season}.csv")
mapping_csv_path = _map_all if os.path.exists(_map_all) else _map_curr

# Output folder
output_dir = os.path.join(INPUTS, "Fixtures")
os.makedirs(output_dir, exist_ok=True)

# League list
leagues = {
    "premier_league": "ENG-Premier League",
    "bundesliga": "GER-Bundesliga",
    "serie_a": "ITA-Serie A",
    "la_liga": "ESP-La Liga",
    "ligue_1": "FRA-Ligue 1",
}

# -----------------------------
# HELPERS
# -----------------------------
def parse_uk_date(date_str: str) -> pd.Timestamp:
    """
    Convert dd-mm-yyyy to UTC midnight timestamp.
    """
    return pd.to_datetime(date_str, format="%d-%m-%Y", utc=True)

# Inclusive date range:
# start = 00:00:00 on start date UTC
# end   = 23:59:59 on end date UTC
start_ts = parse_uk_date(start_date_input)
end_ts = parse_uk_date(end_date_input) + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)

# -----------------------------
# LOAD TEAM NAME MAPPING
# -----------------------------
mapping_df = pd.read_csv(mapping_csv_path)

required_cols = {"espn_team", "understat_team"}
missing_cols = required_cols - set(mapping_df.columns)
if missing_cols:
    raise ValueError(f"Mapping CSV is missing required columns: {missing_cols}")

TEAM_NAME_MAP = dict(zip(mapping_df["espn_team"], mapping_df["understat_team"]))

print(f"Loaded {len(TEAM_NAME_MAP)} team mappings from:")
print(mapping_csv_path)
print()
print(f"Filtering fixtures from {start_date_input} to {end_date_input} inclusive")
print(f"Internal UTC range: {start_ts} to {end_ts}")
print()

# -----------------------------
# PULL FIXTURES, MAP NAMES, SAVE
# -----------------------------
for file_name, league in leagues.items():
    print(f"Downloading {league}")

    espn = with_retry(sd.ESPN, leagues=league, seasons=season, data_dir=SOCCERDATA_CACHE)
    df   = with_retry(espn.read_schedule)
    df   = df.reset_index()

    # Parse datetime
    df["date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")
    df = df.dropna(subset=["date", "home_team", "away_team"]).copy()

    # Filter to inclusive date range
    df = df[(df["date"] >= start_ts) & (df["date"] <= end_ts)].copy()

    if df.empty:
        print(f"No fixtures found for {league} in this date range.")
        print()
        continue

    # Check unmapped names before mapping
    raw_teams = sorted(set(df["home_team"]).union(set(df["away_team"])))
    unmapped_teams = [team for team in raw_teams if team not in TEAM_NAME_MAP]

    if unmapped_teams:
        print(f"WARNING: Unmapped team names found for {league}:")
        for team in unmapped_teams:
            print(f"  - {team}")

    # Map ESPN names to Understat names
    df["home_team"] = df["home_team"].map(lambda x: TEAM_NAME_MAP.get(x, x))
    df["away_team"] = df["away_team"].map(lambda x: TEAM_NAME_MAP.get(x, x))

    # Build output
    fixtures = pd.DataFrame({
        "Date": df["date"].dt.strftime("%Y-%m-%d"),
        "Home Team": df["home_team"],
        "Away Team": df["away_team"],
        "Start Time (UTC)": df["date"].dt.strftime("%H:%M:%S"),
    })

    fixtures = fixtures.sort_values(
        ["Date", "Start Time (UTC)", "Home Team", "Away Team"]
    ).reset_index(drop=True)

    # Save
    path = os.path.join(output_dir, f"{file_name}_fixtures.csv")
    fixtures.to_csv(path, index=False)

    print(f"Saved {path}")
    print(fixtures.head(10))
    print("-" * 60)

print("Done.")

Loaded 166 team mappings from:
/Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/Mapping/Club_Mapping_All.csv

Filtering fixtures from 15-05-2026 to 19-05-2026 inclusive
Internal UTC range: 2026-05-15 00:00:00+00:00 to 2026-05-19 23:59:59+00:00



[05/17/26 16:33:57] INFO     Saving cached data to                                                   ]8;id=15574468;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=15574469;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\
                             /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/soccerda               
                             ta_cache                                                                              

[2026-05-17 16:33:57] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: /Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/tls_requests/bin/tls-client-darwin-arm64-1.13.1.dylib


                    INFO     Successfully loaded TLS library:                                      ]8;id=15574476;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/tls_requests/models/libraries.py\libraries.py]8;;\:]8;id=15574477;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/tls_requests/models/libraries.py#397\397]8;;\
                             /Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.1                 
                             2/site-packages/tls_requests/bin/tls-client-darwin-arm64-1.13.1.dylib                 

Saved /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/Fixtures/premier_league_fixtures.csv
         Date                Home Team          Away Team Start Time (UTC)
0  2026-05-15              Aston Villa          Liverpool         19:00:00
1  2026-05-17        Manchester United  Nottingham Forest         11:30:00
2  2026-05-17                Brentford     Crystal Palace         14:00:00
3  2026-05-17                  Everton         Sunderland         14:00:00
4  2026-05-17                    Leeds           Brighton         14:00:00
5  2026-05-17  Wolverhampton Wanderers             Fulham         14:00:00
6  2026-05-17         Newcastle United           West Ham         16:30:00
7  2026-05-18                  Arsenal            Burnley         19:00:00
8  2026-05-19              Bournemouth    Manchester City         18:30:00
9  2026-05-19                  Chelsea          Tottenham         19:15:00
------------------------------------------------------------


...

Saved /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/Fixtures/serie_a_fixtures.csv
         Date Home Team          Away Team Start Time (UTC)
0  2026-05-17      Como  Parma Calcio 1913         10:00:00
1  2026-05-17     Genoa           AC Milan         10:00:00
2  2026-05-17  Juventus         Fiorentina         10:00:00
3  2026-05-17      Pisa             Napoli         10:00:00
4  2026-05-17      Roma              Lazio         10:00:00
5  2026-05-17     Inter             Verona         13:00:00
6  2026-05-17  Atalanta            Bologna         16:00:00
7  2026-05-17  Cagliari             Torino         18:45:00
8  2026-05-17  Sassuolo              Lecce         18:45:00
9  2026-05-17   Udinese          Cremonese         18:45:00
------------------------------------------------------------


[05/17/26 16:35:18] INFO     Saving cached data to                                                   ]8;id=15574494;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=15574495;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\
                             /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/soccerda               
                             ta_cache                                                                              

Saved /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/Fixtures/la_liga_fixtures.csv
         Date        Home Team    Away Team Start Time (UTC)
0  2026-05-17    Athletic Club   Celta Vigo         17:00:00
1  2026-05-17  Atletico Madrid       Girona         17:00:00
2  2026-05-17            Elche       Getafe         17:00:00
3  2026-05-17          Levante     Mallorca         17:00:00
4  2026-05-17          Osasuna     Espanyol         17:00:00
5  2026-05-17   Rayo Vallecano   Villarreal         17:00:00
6  2026-05-17      Real Oviedo       Alaves         17:00:00
7  2026-05-17    Real Sociedad     Valencia         17:00:00
8  2026-05-17          Sevilla  Real Madrid         17:00:00
9  2026-05-17        Barcelona   Real Betis         19:15:00
------------------------------------------------------------


[05/17/26 16:35:50] INFO     Saving cached data to                                                   ]8;id=15574500;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=15574501;file:///Users/patrickknott/Desktop/Football_Prediction_Project/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\
                             /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/soccerda               
                             ta_cache                                                                              

Saved /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/Fixtures/ligue_1_fixtures.csv
         Date   Home Team            Away Team Start Time (UTC)
0  2026-05-17       Brest               Angers         19:00:00
1  2026-05-17       Lille              Auxerre         19:00:00
2  2026-05-17     Lorient             Le Havre         19:00:00
3  2026-05-17        Lyon                 Lens         19:00:00
4  2026-05-17   Marseille               Rennes         19:00:00
5  2026-05-17      Nantes             Toulouse         19:00:00
6  2026-05-17        Nice                 Metz         19:00:00
7  2026-05-17    Paris FC  Paris Saint Germain         19:00:00
8  2026-05-17  Strasbourg               Monaco         19:00:00
------------------------------------------------------------
Done.


# **Past Fixtures Comp Agnostic**

In [6]:
import requests
import pandas as pd
import time

# -----------------------------------------------------------------------
# CONFIG
# -----------------------------------------------------------------------
MAPPING_PATH = os.path.join(INPUTS, "Mapping", "Club_Mapping_All.csv")
ID_CACHE_PATH = os.path.join(INPUTS, "Mapping", "ESPN_Team_IDs.csv")

# All league tiers to search — ordered by likelihood
LEAGUE_SLUGS_ALL_TIERS = [
    "eng.1", "eng.2", "eng.3", "eng.4",
    "esp.1", "esp.2",
    "ger.1", "ger.2",
    "ita.1", "ita.2",
    "fra.1", "fra.2",
]

SEASONS = list(range(2025, 2013, -1))  # 2025 back to 2014

# -----------------------------------------------------------------------
# Load mapping — get the ESPN names we need IDs for
# -----------------------------------------------------------------------
mapping_df  = pd.read_csv(MAPPING_PATH)
mapping_df  = mapping_df.dropna(subset=["espn_team"])
target_names = set(mapping_df["espn_team"].unique())
print(f"Total ESPN team names to resolve: {len(target_names)}")

# -----------------------------------------------------------------------
# ID resolution
# -----------------------------------------------------------------------
name_to_id = {}  # espn_name → team_id

def fetch_ids_for_slug_season(slug, season):
    url = f"http://site.api.espn.com/apis/site/v2/sports/soccer/{slug}/teams?season={season}"
    def _fetch():
        r     = requests.get(url, timeout=10)
        data  = r.json()
        teams = data["sports"][0]["leagues"][0]["teams"]
        return {t["team"]["displayName"]: t["team"]["id"] for t in teams}
    try:
        return with_retry(_fetch)
    except Exception:
        return {}

# First pass — current season across all tiers
print("\nPass 1: Current season across all tiers...")
for slug in LEAGUE_SLUGS_ALL_TIERS:
    ids = fetch_ids_for_slug_season(slug, 2025)
    for name, tid in ids.items():
        if name in target_names and name not in name_to_id:
            name_to_id[name] = tid
    time.sleep(0.2)

print(f"Resolved after pass 1: {len(name_to_id)}/{len(target_names)}")

# Second pass — go back season by season for unresolved teams
unresolved = target_names - set(name_to_id.keys())
print(f"\nPass 2: {len(unresolved)} teams unresolved, going back through seasons...")

for season in SEASONS[1:]:  # skip 2025, already done
    if not unresolved:
        break
    print(f"  Season {season} — {len(unresolved)} still unresolved")
    for slug in LEAGUE_SLUGS_ALL_TIERS:
        if not unresolved:
            break
        ids = fetch_ids_for_slug_season(slug, season)
        for name, tid in ids.items():
            if name in unresolved:
                name_to_id[name] = tid
                unresolved.discard(name)
        time.sleep(0.2)

# -----------------------------------------------------------------------
# Report and save
# -----------------------------------------------------------------------
print(f"\nFinal resolved: {len(name_to_id)}/{len(target_names)}")

if unresolved:
    print(f"Still unresolved ({len(unresolved)}):")
    for name in sorted(unresolved):
        print(f"  {name}")

id_df = pd.DataFrame([
    {"espn_team": name, "espn_id": tid}
    for name, tid in name_to_id.items()
])
id_df.to_csv(ID_CACHE_PATH, index=False)
print(f"\nSaved ID cache: {ID_CACHE_PATH}")

Total ESPN team names to resolve: 166

Pass 1: Current season across all tiers...
Resolved after pass 1: 152/166

Pass 2: 14 teams unresolved, going back through seasons...
  Season 2024 — 14 still unresolved
  Season 2023 — 14 still unresolved
  Season 2022 — 14 still unresolved
  Season 2021 — 14 still unresolved
  Season 2020 — 14 still unresolved
  Season 2019 — 14 still unresolved
  Season 2018 — 14 still unresolved
  Season 2017 — 14 still unresolved
  Season 2016 — 14 still unresolved
  Season 2015 — 14 still unresolved
  Season 2014 — 14 still unresolved

Final resolved: 152/166
Still unresolved (14):
  AC Ajaccio
  Benevento
  Bordeaux
  Brescia
  Caen
  Chievo Verona
  Crotone
  Dijon FCO
  Evian Thonon Gaillard
  FC Ingolstadt 04
  GFC Ajaccio
  Nimes
  SPAL
  Salernitana

Saved ID cache: /Users/patrickknott/Desktop/Football_Prediction_Project/Inputs/Mapping/ESPN_Team_IDs.csv


In [7]:
import requests
import pandas as pd
import time

# -----------------------------------------------------------------------
# CONFIG — edit here only
# -----------------------------------------------------------------------
CURRENT_SEASON = "2025-26"
MODE           = "All"        # "All" or "Current"

MAPPING_PATH   = os.path.join(INPUTS, "Mapping", "Club_Mapping_All.csv")
ID_CACHE_PATH  = os.path.join(INPUTS, "Mapping", "ESPN_Team_IDs.csv")
OUT_DIR        = os.path.join(INPUTS, "Fixtures")

# -----------------------------------------------------------------------
# Parse season config
# -----------------------------------------------------------------------
current_season_espn  = int(CURRENT_SEASON.split("-")[0])
current_season_label = CURRENT_SEASON

if MODE == "Current":
    SEASONS_HISTORIC = []
    SEASONS_CURRENT  = [current_season_espn]
else:
    SEASONS_HISTORIC = list(range(2014, current_season_espn))
    SEASONS_CURRENT  = [current_season_espn]

print(f"Mode: {MODE}")
print(f"Historic seasons: {SEASONS_HISTORIC}")
print(f"Current season:   {SEASONS_CURRENT}")

# -----------------------------------------------------------------------
# Load mapping and ID cache
# -----------------------------------------------------------------------
mapping_df  = pd.read_csv(MAPPING_PATH).dropna(subset=["espn_team"])
espn_to_und = dict(zip(mapping_df["espn_team"], mapping_df["understat_team"]))

id_df       = pd.read_csv(ID_CACHE_PATH)
id_df["espn_id"] = id_df["espn_id"].astype(str)
name_to_id  = dict(zip(id_df["espn_team"], id_df["espn_id"]))

# Build all_teams from ID cache — id → espn_name
all_teams = {tid: name for name, tid in name_to_id.items() if name in espn_to_und}

print(f"Teams in mapping:   {len(espn_to_und)}")
print(f"Teams with ESPN ID: {len(all_teams)}")

missing = set(espn_to_und.keys()) - set(name_to_id.keys())
if missing:
    print(f"No ID found for ({len(missing)}) — skipped: {sorted(missing)}")

# -----------------------------------------------------------------------
# Pull and parse schedule
# -----------------------------------------------------------------------
def get_team_schedule(team_id, season_year):
    url = (
        f"http://site.api.espn.com/apis/site/v2/sports/soccer/all/teams"
        f"/{team_id}/schedule?season={season_year}"
    )
    def _fetch():
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        return r.json()
    try:
        return with_retry(_fetch)
    except Exception:
        return None

def parse_team_schedule(data, team_espn_name):
    if not data:
        return []

    understat_name = espn_to_und[team_espn_name]
    today          = pd.Timestamp.today().normalize()
    rows           = []

    for event in data.get("events", []):
        try:
            date = pd.Timestamp(event["date"][:10])
            if date > today:
                continue

            competitors = event["competitions"][0]["competitors"]
            team_entry  = next(
                (c for c in competitors
                 if c["team"].get("displayName") == team_espn_name),
                None
            )
            if team_entry is None:
                continue

            venue = team_entry.get("homeAway", "unknown")

            rows.append({
                "date":  date,
                "team":  understat_name,
                "venue": venue,
            })
        except Exception:
            continue

    return rows

# -----------------------------------------------------------------------
# Pull loop
# -----------------------------------------------------------------------
def pull_seasons(seasons, label):
    records = []
    total   = len(all_teams) * len(seasons)
    count   = 0

    for team_id, team_espn_name in all_teams.items():
        for season_year in seasons:
            count += 1
            data = get_team_schedule(team_id, season_year)
            rows = parse_team_schedule(data, team_espn_name)
            records.extend(rows)
            if count % 50 == 0:
                print(f"[{label}] [{count}/{total}] {team_espn_name} {season_year} — {len(rows)} dates")
                if records:
                    _cp = os.path.join(OUT_DIR, f"_checkpoint_{label.lower()}.csv")
                    pd.DataFrame(records).to_csv(_cp, index=False)
            time.sleep(0.3)

    if not records:
        return pd.DataFrame(columns=["date", "team", "venue"])
    df = pd.DataFrame(records)
    df = df.drop_duplicates(["date", "team"]).reset_index(drop=True)
    df = df.sort_values(["team", "date"]).reset_index(drop=True)
    return df

# -----------------------------------------------------------------------
# Run and save
# -----------------------------------------------------------------------
if MODE == "Current":
    current_df = pull_seasons(SEASONS_CURRENT, "Current")
    out_path   = f"{OUT_DIR}/all_competitions_fixtures_{current_season_label}.csv"
    current_df.to_csv(out_path, index=False)
    print(f"\nSaved: {out_path} ({len(current_df)} records)")

else:
    print("\n--- Pulling historic seasons ---")
    historic_df = pull_seasons(SEASONS_HISTORIC, "Historic")
    hist_path   = f"{OUT_DIR}/historic_competitions_fixtures.csv"
    historic_df.to_csv(hist_path, index=False)
    print(f"Saved: {hist_path} ({len(historic_df)} records)")

    print("\n--- Pulling current season ---")
    current_df = pull_seasons(SEASONS_CURRENT, "Current")
    curr_path  = f"{OUT_DIR}/all_competitions_fixtures_{current_season_label}.csv"
    current_df.to_csv(curr_path, index=False)
    print(f"Saved: {curr_path} ({len(current_df)} records)")

print("\nDone.")

Mode: All
Historic seasons: [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Current season:   [2025]
Teams in mapping:   166
Teams with ESPN ID: 152
No ID found for (14) — skipped: ['AC Ajaccio', 'Benevento', 'Bordeaux', 'Brescia', 'Caen', 'Chievo Verona', 'Crotone', 'Dijon FCO', 'Evian Thonon Gaillard', 'FC Ingolstadt 04', 'GFC Ajaccio', 'Nimes', 'SPAL', 'Salernitana']

--- Pulling historic seasons ---
[Historic] [50/1672] Brighton & Hove Albion 2019 — 41 dates
[Historic] [100/1672] Fulham 2014 — 53 dates
[Historic] [150/1672] Manchester United 2020 — 62 dates
[Historic] [200/1672] West Ham United 2015 — 55 dates
[Historic] [250/1672] Leicester City 2021 — 60 dates
[Historic] [300/1672] Southampton 2016 — 54 dates
[Historic] [350/1672] West Bromwich Albion 2022 — 51 dates
[Historic] [400/1672] Athletic Club 2017 — 55 dates
[Historic] [450/1672] Elche 2023 — 46 dates
[Historic] [500/1672] Mallorca 2018 — 48 dates
[Historic] [550/1672] Real Madrid 2024 — 65 dates
[His